# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mohamedahmed02/Flyrank-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*
I will use Logistic Regression because this lane has a binary observed outcome: whether organic clicks increase in April compared with March.

The model is simple and interpretable, and its predicted probability can be used to rank content items for review. This fits the decision question better than using predicted class labels directly.

I will start with this simple model before trying a more complex model because the goal is useful decision support, not complexity by itself.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# W05 — Build March features + April label
# ============================================================

!pip -q install -U duckdb huggingface_hub pyarrow scikit-learn

import os
import duckdb
import numpy as np
import pandas as pd

from google.colab import userdata
from sklearn.model_selection import GroupShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.inspection import permutation_importance

SEED = 42
K = 50

HF_TOKEN = userdata.get("HF_TOKEN")

if not HF_TOKEN:
    raise RuntimeError("HF_TOKEN was not found in Colab Secrets.")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf_secret (
    TYPE HTTP,
    EXTRA_HTTP_HEADERS MAP {{
        'Authorization': 'Bearer {HF_TOKEN}'
    }}
);
""")

MARCH_URL = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month=2026-03/data_0.parquet"
)

APRIL_URL = (
    "https://huggingface.co/datasets/"
    "FlyRank/internship-warehouse/resolve/main/"
    "fact_content_daily_performance/month=2026-04/data_0.parquet"
)

# March = decision-time features
# April = future outcome / label
dataset = con.execute(f"""
WITH march AS (
    SELECT
        client_hash_id,
        content_hash_id,

        SUM(gsc_clicks) AS gsc_clicks,
        SUM(gsc_impressions) AS gsc_impressions,

        CASE
            WHEN SUM(gsc_impressions) > 0
            THEN CAST(SUM(gsc_sum_position) AS DOUBLE)
                 / SUM(gsc_impressions)
            ELSE NULL
        END AS gsc_avg_position,

        SUM(ga4_sessions) AS ga4_sessions,
        SUM(scroll_events) AS scroll_events

    FROM read_parquet('{MARCH_URL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
),

april AS (
    SELECT
        client_hash_id,
        content_hash_id,
        SUM(gsc_clicks) AS april_clicks
    FROM read_parquet('{APRIL_URL}')
    WHERE gsc_data_available IS TRUE
    GROUP BY client_hash_id, content_hash_id
)

SELECT
    m.*,
    a.april_clicks,

    CASE
        WHEN a.april_clicks > m.gsc_clicks THEN 1
        ELSE 0
    END AS label

FROM march m
INNER JOIN april a
    ON m.client_hash_id = a.client_hash_id
   AND m.content_hash_id = a.content_hash_id
""").fetchdf()

print("Rows:", len(dataset))
print("Positive rate:", dataset["label"].mean())
display(dataset.head())

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.5/21.5 MB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 793.2/793.2 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 9.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.2 MB/s eta 0:00:00


FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Rows: 158549
Positive rate: 0.17622943064919994


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events,april_clicks,label
0,client_62f4a7e64f5e0096,content_39d7361b4945d504,0.0,77.0,4.311688,NaN,NaN,0.0,0
1,client_62f4a7e64f5e0096,content_c03ecafd4c999f15,22.0,10849.0,8.049866,NaN,NaN,23.0,1
2,client_62f4a7e64f5e0096,content_e689bc511192751a,0.0,61.0,5.885246,NaN,NaN,1.0,1
3,client_62f4a7e64f5e0096,content_7dbc094b799e05a4,1.0,705.0,5.863830,NaN,NaN,0.0,0
4,client_62f4a7e64f5e0096,content_40b10da45f4c1cb5,0.0,50.0,14.360000,NaN,NaN,0.0,0


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*
I will use an 80/20 grouped split by client_hash_id.

Grouping by client prevents content from the same client from appearing in both training and test sets. This reduces the risk that the model learns client-specific patterns that would not generalize to unseen clients.

The split is fixed with a random seed so the result is reproducible.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Grouped train/test split
# ============================================================

FEATURES = [
    "gsc_clicks",
    "gsc_impressions",
    "gsc_avg_position",
    "ga4_sessions",
    "scroll_events",
]

TARGET = "label"
GROUP = "client_hash_id"

X = dataset[FEATURES].copy()
y = dataset[TARGET].astype(int)
groups = dataset[GROUP]

splitter = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=SEED
)

train_idx, test_idx = next(
    splitter.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx].copy()
X_test = X.iloc[test_idx].copy()

y_train = y.iloc[train_idx].copy()
y_test = y.iloc[test_idx].copy()

test_rows = dataset.iloc[test_idx].copy()

print("Train rows:", len(X_train))
print("Test rows:", len(X_test))
print("Train clients:", dataset.iloc[train_idx][GROUP].nunique())
print("Test clients:", dataset.iloc[test_idx][GROUP].nunique())
print("Client overlap:",
      len(
          set(dataset.iloc[train_idx][GROUP])
          & set(dataset.iloc[test_idx][GROUP])
      ))

Train rows: 137445
Test rows: 21104
Train clients: 36
Test clients: 10
Client overlap: 0


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Logistic Regression
# ============================================================

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(
        max_iter=1000,
        random_state=SEED
    ))
])

model.fit(X_train, y_train)

test_rows["model_probability"] = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

# ============================================================
# Precision@K
# ============================================================

def precision_at_k(y_true, scores, k=50):
    y_true = np.asarray(y_true)
    scores = np.asarray(scores)

    top_idx = np.argsort(-scores)[:k]

    return y_true[top_idx].mean()


model_p50 = precision_at_k(
    y_test,
    test_rows["model_probability"],
    K
)

base_rate = y_test.mean()

print(f"Model Precision@{K}: {model_p50:.4f}")
print(f"Test base rate: {base_rate:.4f}")

# ============================================================
# Week-4 rule baseline applied to the same W05 test split
# ============================================================

baseline_test = test_rows.copy()

baseline_test["ctr"] = np.where(
    baseline_test["gsc_impressions"] > 0,
    baseline_test["gsc_clicks"] / baseline_test["gsc_impressions"],
    np.nan
)

baseline_test["volume_score"] = np.select(
    [
        baseline_test["gsc_impressions"] >= 10000,
        baseline_test["gsc_impressions"] >= 1000,
        baseline_test["gsc_impressions"] >= 100,
        baseline_test["gsc_impressions"] >= 1,
    ],
    [5, 4, 3, 2],
    default=0
)

baseline_test["ctr_position_score"] = np.where(
    (
        baseline_test["gsc_avg_position"].between(4, 20)
        & (baseline_test["ctr"] < 0.01)
    ),
    5,
    0
)

baseline_test["baseline_score"] = (
    baseline_test["volume_score"]
    + baseline_test["ctr_position_score"]
)

baseline_p50 = precision_at_k(
    baseline_test["label"],
    baseline_test["baseline_score"],
    K
)

print(f"Baseline Precision@{K}: {baseline_p50:.4f}")

# ============================================================
# Required comparison table
# ============================================================

comparison = pd.DataFrame({
    "method": [
        "Week-4 rule baseline",
        "Logistic Regression"
    ],
    "precision_at_50": [
        baseline_p50,
        model_p50
    ],
    "test_base_rate": [
        base_rate,
        base_rate
    ]
})

comparison


Model trained successfully.
Model Precision@50: 0.3800
Test base rate: 0.1641
Baseline Precision@50: 0.1200


,method,precision_at_50,test_base_rate
0,Week-4 rule baseline,0.12,0.164139
1,Logistic Regression,0.38,0.164139


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

I compare the model with the Week-4 rule baseline on the same held-out clients and the same Precision@50 metric.

The error analysis focuses on the highest-ranked false positives and false negatives. I also inspect permutation importance to understand which March features contribute most to the model's predictions.

The model should only be considered an improvement if it improves the decision metric without relying on suspicious or future information.

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ============================================================
# Error analysis
# ============================================================

test_rows["prediction"] = (
    test_rows["model_probability"] >= 0.5
).astype(int)

test_rows["error"] = (
    test_rows["prediction"] != test_rows["label"]
)

errors = test_rows[test_rows["error"]].copy()

print("Total test errors:", len(errors))

# Three concrete false positives
false_positives = (
    errors[errors["prediction"] == 1]
    .sort_values("model_probability", ascending=False)
    .head(3)
)

# Three concrete false negatives
false_negatives = (
    errors[errors["prediction"] == 0]
    .sort_values("model_probability", ascending=False)
    .head(3)
)

print("False positives:")
display(
    false_positives[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_clicks",
            "gsc_impressions",
            "gsc_avg_position",
            "ga4_sessions",
            "scroll_events",
            "model_probability",
            "label"
        ]
    ]
)

print("False negatives:")
display(
    false_negatives[
        [
            "client_hash_id",
            "content_hash_id",
            "gsc_clicks",
            "gsc_impressions",
            "gsc_avg_position",
            "ga4_sessions",
            "scroll_events",
            "model_probability",
            "label"
        ]
    ]
)

# ============================================================
# Permutation importance
# ============================================================

perm = permutation_importance(
    model,
    X_test,
    y_test,
    n_repeats=3,
    random_state=SEED,
    scoring="average_precision"
)

importance = (
    pd.DataFrame({
        "feature": FEATURES,
        "importance": perm.importances_mean
    })
    .sort_values("importance", ascending=False)
)

display(importance)

Total test errors: 3477
False positives:


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events,model_probability,label
92354,client_e5c2aa26a8598242,content_b116498610b4e6da,11.0,4591.0,8.717055,1060.0,7.0,0.964505,0
3594,client_fef1a8f436438636,content_6b4ba5a247ea6100,506.0,74334.0,3.960691,650.0,63.0,0.956049,0
65660,client_e5c2aa26a8598242,content_fbbf1f682ae5c5da,453.0,56482.0,4.569296,381.0,6.0,0.829056,0


False negatives:


,client_hash_id,content_hash_id,gsc_clicks,gsc_impressions,gsc_avg_position,ga4_sessions,scroll_events,model_probability,label
63269,client_e5c2aa26a8598242,content_c638f20a9f2e23c9,125.0,27348.0,3.694201,154.0,0.0,0.498443,1
92439,client_e5c2aa26a8598242,content_f07d2b3489009cb3,86.0,34533.0,7.286566,108.0,2.0,0.491569,1
94963,client_e5c2aa26a8598242,content_8456cf958551f365,221.0,21962.0,3.680949,201.0,14.0,0.477039,1


,feature,importance
2,gsc_avg_position,0.034865
1,gsc_impressions,0.008195
4,scroll_events,0.004462
3,ga4_sessions,0.000307
0,gsc_clicks,-0.001138


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

The Logistic Regression model achieved a Precision@50 of 0.38 on the held-out test set, compared with 0.12 for the Week-4 rule baseline. The test-set positive rate was 0.164, so the model's top-50 ranking performed above the overall positive rate and above the baseline.

The model made 3,477 classification errors on the test set. The false-positive examples show cases where the model assigned high probabilities but the April outcome did not increase. The false-negative examples were mostly borderline cases with probabilities close to 0.5.

Permutation importance shows that gsc_avg_position was the strongest feature, followed by gsc_impressions and scroll_events. These results are directional rather than causal.

Overall, the Logistic Regression model provides stronger decision-support ranking than the Week-4 rule baseline on the same held-out clients and Precision@50 metric.